In [9]:
from langgraph.graph import StateGraph , START , END
from typing import TypedDict,Literal
from pydantic import BaseModel,Field
from llm import llm

In [27]:
class State(TypedDict):
    product:str
    review:str
    sentiment:Literal["positive","negative"]
    diagnosis:dict
    response:str

In [29]:
class Schema(BaseModel):
    sentiment: Literal["positive","negative"] = Field(description="Sentiment of the review")

llm1=llm.with_structured_output(Schema) 

class Schema2(BaseModel):
    issue : str = Field(description="What is the issue related to")
    tone : str = Field (description="What is the tone of the review")
llm2=llm.with_structured_output(Schema2)

In [67]:
def createReview(state:State):
    prompt=f"You are a customer , write random review on products (positive) product - {state['product']}"
    res=llm.invoke(prompt).content
    state["review"]=res
    return state

def classifyReview(state:State):
    prompt=f"Specify wether the review is positive or not review - {state['review']}"
    res=llm1.invoke(prompt)
    print(state)
    state['sentiment']=res.sentiment
    return state
def condition(state:State)-> Literal['diagnosis','positive']:
    if(state['sentiment']=="positive"):
        return "positive"
    else : 
        return "diagnosis"
def diagnose(state:State):
    prompt=f"Diagnose the give review - {state['review']}"
    res=llm2.invoke(prompt)
    return {"diagnosis":res}
def negative(state:State):
    prompt=f"Write a response for the review - {state['review']} , diagnosis of review {state['diagnosis']}"
    res=llm.invoke(prompt).content
    return {"response":res}
def positive(state:State):
    prompt=f"Write a response for the review {state['review']}"
    res=llm.invoke(prompt).content
    return {"response":res}


In [72]:
graph=StateGraph(State)

graph.add_node("review",createReview)
graph.add_node("sentiment",classifyReview)

graph.add_node("diagnosis",diagnose)
graph.add_node("negative", negative)

graph.add_node("positive", positive)

graph.add_edge(START,"review")
graph.add_edge("review","sentiment")

graph.add_conditional_edges("sentiment",condition)
graph.add_edge("diagnosis","negative")
graph.add_edge("negative",END)
graph.add_edge("positive",END)
workflow=graph.compile()


In [74]:
res=workflow.invoke({"product": "Car"})
print(res['product'])
print(res['sentiment'])
print(res['response'])


{'product': 'Car', 'review': "**5/5 Stars**\n\nI am absolutely thrilled with my new car. I recently purchased a sleek and powerful sedan, and I must say, it's been a game-changer for me. The moment I stepped inside, I knew I had made the right decision. The interior is luxurious, with premium leather seats and a state-of-the-art infotainment system that makes every drive a pleasure.\n\nThe car's performance is impressive, with a smooth and quiet ride that makes me feel like I'm gliding over the road. The acceleration is instant, and the handling is precise, making it a joy to drive on both city streets and highways.\n\nOne of the features that really stands out to me is the advanced safety features. The car comes equipped with lane departure warning, blind spot detection, and forward collision alert, which gives me peace of mind when driving, especially on long trips.\n\nI've also been impressed with the car's fuel efficiency. Despite its powerful engine, it's been averaging an impress